In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from IPython.display import display

import sys
sys.path.append("..")

from src.common.config import BUCKET_GOLD
from src.common.minio_client import client

sns.set_theme(style="whitegrid")
print("Imports OK")

In [ ]:
def lire_gold(nom_fichier: str) -> pd.DataFrame:
    """Lit un fichier Parquet Gold depuis MinIO."""
    response = client.get_object(BUCKET_GOLD, f"velos/{nom_fichier}.parquet")
    return pd.read_parquet(io.BytesIO(response.read()))

# Chargement de tous les indicateurs
taux_stations    = lire_gold("taux_par_station")
stations_vides   = lire_gold("stations_vides")
stations_saturees= lire_gold("stations_saturees")
pics_horaires    = lire_gold("pics_horaires")
desequilibre     = lire_gold("desequilibre_ville")

print(f"Stations chargées     : {len(taux_stations)}")
print(f"Stations vides        : {len(stations_vides)}")
print(f"Stations saturées     : {len(stations_saturees)}")
print(taux_stations.head())

In [ ]:
top20 = taux_stations.head(20)

plt.figure(figsize=(12, 6))
sns.barplot(data=top20, x="taux_moyen", y="station_name", palette="Reds_r")
plt.title("Top 20 stations – Taux d'utilisation moyen")
plt.xlabel("Taux d'utilisation (0 = vide, 1 = plein)")
plt.ylabel("Station")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Stations quasi-vides (< 10%) : {len(stations_vides)}")
print(f"Stations saturées   (> 90%) : {len(stations_saturees)}")

# Top 10 stations les plus souvent vides
top_vides = stations_vides.head(10)

plt.figure(figsize=(12, 5))
sns.barplot(data=top_vides, x="nb_alertes_vide", y="station_name", palette="Oranges_r")
plt.title("Top 10 stations les plus souvent vides")
plt.xlabel("Nombre d'alertes (snapshot où vélos < 10%)")
plt.ylabel("Station")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=pics_horaires, x="heure", y="bikes_moyen", marker="o", color="steelblue")
plt.title("Disponibilité moyenne des vélos par heure de la journée")
plt.xlabel("Heure (UTC)")
plt.ylabel("Vélos disponibles en moyenne")
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Taux moyen par ville
sns.barplot(data=desequilibre.sort_values("taux_moyen", ascending=False),
            x="ville", y="taux_moyen", palette="Blues_d", ax=axes[0])
axes[0].set_title("Taux d'utilisation moyen par ville")
axes[0].set_xlabel("Ville")
axes[0].set_ylabel("Taux moyen")
axes[0].tick_params(axis="x", rotation=45)

# Nombre de stations par ville
sns.barplot(data=desequilibre.sort_values("nb_stations", ascending=False),
            x="ville", y="nb_stations", palette="Greens_d", ax=axes[1])
axes[1].set_title("Nombre de stations par ville")
axes[1].set_xlabel("Ville")
axes[1].set_ylabel("Nb stations")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Recharge le Silver pour avoir les coordonnées GPS
from src.common.minio_client import list_objects

fichiers_silver = list_objects("urbanhub-silver", prefix="velos/")
dfs = []
for f in fichiers_silver:
    resp = client.get_object("urbanhub-silver", f)
    dfs.append(pd.read_parquet(io.BytesIO(resp.read())))

df_silver = pd.concat(dfs, ignore_index=True)
df_silver["taux"] = df_silver["bikes_available"] / df_silver["total_docks"]

# Moyenne par station
stations_map = df_silver.groupby(["station_id", "station_name", "latitude", "longitude"]).agg(
    taux_moyen=("taux", "mean")
).reset_index()

# Carte centrée sur la France
carte = folium.Map(location=[46.8, 2.3], zoom_start=6)

for _, row in stations_map.iterrows():
    # Rouge = vide, Vert = OK, Orange = saturé
    if row["taux_moyen"] < 0.10:
        couleur = "red"
    elif row["taux_moyen"] > 0.90:
        couleur = "orange"
    else:
        couleur = "green"

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color=couleur,
        fill=True,
        popup=f"{row['station_name']} – taux: {row['taux_moyen']:.0%}",
    ).add_to(carte)

carte

In [ ]:
# Recharge le Silver pour avoir les coordonnées GPS
from src.common.minio_client import list_objects

fichiers_silver = list_objects("urbanhub-silver", prefix="velos/")
dfs = []
for f in fichiers_silver:
    resp = client.get_object("urbanhub-silver", f)
    dfs.append(pd.read_parquet(io.BytesIO(resp.read())))

df_silver = pd.concat(dfs, ignore_index=True)
df_silver["taux"] = df_silver["bikes_available"] / df_silver["total_docks"]

# Moyenne par station
stations_map = df_silver.groupby(["station_id", "station_name", "latitude", "longitude"]).agg(
    taux_moyen=("taux", "mean")
).reset_index()

# Carte centrée sur la France
carte = folium.Map(location=[46.8, 2.3], zoom_start=6)

for _, row in stations_map.iterrows():
    # Rouge = vide, Vert = OK, Orange = saturé
    if row["taux_moyen"] < 0.10:
        couleur = "red"
    elif row["taux_moyen"] > 0.90:
        couleur = "orange"
    else:
        couleur = "green"

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color=couleur,
        fill=True,
        popup=f"{row['station_name']} – taux: {row['taux_moyen']:.0%}",
    ).add_to(carte)

carte